In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import time
import re
from tqdm.auto import tqdm

In [ ]:
# ==============================
# SETTINGS
# ==============================

INPUT_FILE = "/content/drive/MyDrive/AI Project/2023/Pubmed Exports_2023_Final.csv"

OUTPUT_FILE = "/content/drive/MyDrive/AI Project/2023/2023 Prostate Abstract Matches.xlsx"

BATCH_SIZE = 200

SEARCH_TERMS = [
    "prostate cancer",
    "prostate carcinoma",
    "prostate neoplasm"
]

# NCBI asks that E-utilities requests identify the user/tool.
# Put your own email here.
NCBI_EMAIL = "YOUR_EMAIL@brown.edu"

# Optional. Leave blank if you do not have an NCBI API key.
NCBI_API_KEY = ""

In [ ]:
df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df):,} articles")
print(df.columns.tolist())

df.head()

Loaded 9,183 articles
['entry_number', 'pmid', 'title', 'details', 'author', 'url']


,entry_number,pmid,title,details,author,url
0,44080,36912538,"Fifteen-Year Outcomes after Monitoring, Surger...",N Engl J Med. 2023 Apr 27;388(17):1547-1558. d...,Hamdy FC,https://www.ncbi.nlm.nih.gov/pubmed/36912538
1,44081,37150260,Salvage Radiation Therapy After Radical Prosta...,Int J Radiat Oncol Biol Phys. 2023 Nov 1;117(3...,Petersen PM,https://www.ncbi.nlm.nih.gov/pubmed/37150260
2,44082,36494221,Management of Patients with Advanced Prostate ...,Eur Urol. 2023 Mar;83(3):267-293. doi: 10.1016...,Gillessen S,https://www.ncbi.nlm.nih.gov/pubmed/36494221
3,44083,36591993,Salvage radical prostatectomy,Curr Opin Urol. 2023 Mar 1;33(2):163-167. doi:...,Nabavizadeh R,https://www.ncbi.nlm.nih.gov/pubmed/36591993
4,44084,37315297,Effect of Brachytherapy With External Beam Rad...,J Clin Oncol. 2023 Aug 20;41(24):4035-4044. do...,Michalski JM,https://www.ncbi.nlm.nih.gov/pubmed/37315297


In [ ]:
# If a PMID column already exists, use it.
if "pmid" in df.columns:
    df["pmid"] = (
        df["pmid"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

# Otherwise extract PMID from the PubMed URL.
elif "url" in df.columns:
    df["pmid"] = (
        df["url"]
        .astype(str)
        .str.extract(r"(\d+)(?:/)?$")[0]
    )

else:
    raise ValueError("The CSV needs either a 'pmid' column or a 'url' column.")

# Remove obvious invalid PMIDs
df["pmid"] = df["pmid"].where(df["pmid"].str.fullmatch(r"\d+"))

print(f"Valid PMIDs: {df['pmid'].notna().sum():,}")
print(f"Missing PMIDs: {df['pmid'].isna().sum():,}")

Valid PMIDs: 9,183
Missing PMIDs: 0


In [ ]:
def fetch_pubmed_abstracts(pmids, batch_size=200):

    BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    abstracts = {}

    # Reuse the same HTTP connection
    session = requests.Session()

    valid_pmids = [
        str(pmid)
        for pmid in pmids
        if pd.notna(pmid)
    ]

    for start in tqdm(
        range(0, len(valid_pmids), batch_size),
        desc="Downloading PubMed abstracts"
    ):

        batch = valid_pmids[start:start + batch_size]

        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml",
            "tool": "prostate_abstract_filter",
            "email": NCBI_EMAIL
        }

        if NCBI_API_KEY:
            params["api_key"] = NCBI_API_KEY

        # ----------------------
        # Retry failed requests
        # ----------------------
        success = False

        for attempt in range(5):
            try:
                response = session.get(
                    BASE_URL,
                    params=params,
                    timeout=60
                )

                response.raise_for_status()
                success = True
                break

            except requests.RequestException as e:
                print(
                    f"\nRequest failed "
                    f"(attempt {attempt + 1}/5): {e}"
                )

                time.sleep(2 ** attempt)

        if not success:
            print(
                f"Skipping batch beginning with PMID {batch[0]}"
            )
            continue

        # ----------------------
        # Parse PubMed XML
        # ----------------------
        root = ET.fromstring(response.content)

        for article in root.findall(".//PubmedArticle"):

            pmid_node = article.find(
                ".//MedlineCitation/PMID"
            )

            if pmid_node is None:
                continue

            pmid = pmid_node.text

            abstract_nodes = article.findall(
                ".//Article/Abstract/AbstractText"
            )

            parts = []

            for node in abstract_nodes:

                text = "".join(node.itertext()).strip()

                if not text:
                    continue

                label = node.attrib.get("Label")

                if label:
                    parts.append(
                        f"{label}: {text}"
                    )
                else:
                    parts.append(text)

            abstracts[pmid] = " ".join(parts)

        # ----------------------
        # Respect NCBI rate limit
        # ----------------------
        if NCBI_API_KEY:
            time.sleep(0.11)   # <10 requests/sec
        else:
            time.sleep(0.34)   # <3 requests/sec

    return abstracts

In [ ]:
abstract_dict = fetch_pubmed_abstracts(
    df["pmid"],
    batch_size=BATCH_SIZE
)

print(f"\nRetrieved {len(abstract_dict):,} PubMed records")


Retrieved 9,183 PubMed records


In [ ]:
df["Abstract"] = (
    df["pmid"]
    .map(abstract_dict)
    .fillna("")
)

with_abstract = (df["Abstract"].str.len() > 0).sum()
without_abstract = (df["Abstract"].str.len() == 0).sum()

print(f"Articles with abstracts: {with_abstract:,}")
print(f"Articles without abstracts: {without_abstract:,}")

Articles with abstracts: 8,374
Articles without abstracts: 809


In [ ]:
def find_matched_terms(abstract):

    text = str(abstract).lower()

    matches = [
        term
        for term in SEARCH_TERMS
        if term in text
    ]

    return ", ".join(matches)

In [ ]:
df["Matched Terms"] = df["Abstract"].apply(find_matched_terms)

matches = df[
    df["Matched Terms"] != ""
].copy()

print(f"Total matching articles: {len(matches):,}")

Total matching articles: 5,588


In [ ]:
for term in SEARCH_TERMS:

    count = matches["Abstract"].str.contains(
        re.escape(term),
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f'{term}: {count:,}')

matches[
    ["pmid", "title", "Matched Terms", "Abstract"]
].head(20)

prostate cancer: 5,556
prostate carcinoma: 58
prostate neoplasm: 3


,pmid,title,Matched Terms,Abstract
0,36912538,"Fifteen-Year Outcomes after Monitoring, Surger...",prostate cancer,BACKGROUND: Between 1999 and 2009 in the Unite...
1,37150260,Salvage Radiation Therapy After Radical Prosta...,prostate cancer,PURPOSE: Emerging data indicate comparable dis...
2,36494221,Management of Patients with Advanced Prostate ...,prostate cancer,BACKGROUND: Innovations in imaging and molecul...
4,37315297,Effect of Brachytherapy With External Beam Rad...,prostate cancer,PURPOSE: To determine whether addition of exte...
5,36527579,Comparison of therapeutic features and oncolog...,prostate cancer,OBJECTIVES: To compare the therapeutic feature...
6,36167599,ARNEO: A Randomized Phase II Trial of Neoadjuv...,prostate cancer,BACKGROUND: High-risk prostate cancer (PCa) pa...
8,38027094,The metabolic repression effect of carbon-ion ...,prostate cancer,BACKGROUND: Metastatic prostate cancer (PCa) p...
9,37891072,Administering [(177)Lu]Lu-PSMA-617 Prior to Ra...,prostate cancer,BACKGROUND: High-risk localised prostate cance...
10,37584213,The oncologic risk of magnetic resonance imagi...,prostate cancer,BACKGROUND: Magnetic resonance imaging (MRI)-t...
12,35260794,Long term genitourinary toxicity following cur...,prostate cancer,BACKGROUND: Recent studies have shown that rad...


In [ ]:
no_abstracts = df[
    df["Abstract"].str.strip() == ""
].copy()

print(f"Articles without abstracts: {len(no_abstracts):,}")

Articles without abstracts: 809


In [ ]:
# ==========================================
# CREATE NON-OVERLAPPING TERM GROUPS
# ==========================================

# Check which phrases appear in each matching abstract
has_cancer = matches["Abstract"].str.contains(
    "prostate cancer",
    case=False,
    regex=False,
    na=False
)

has_carcinoma = matches["Abstract"].str.contains(
    "prostate carcinoma",
    case=False,
    regex=False,
    na=False
)

has_neoplasm = matches["Abstract"].str.contains(
    "prostate neoplasm",
    case=False,
    regex=False,
    na=False
)

# Count how many of the three phrases each article contains
term_count = (
    has_cancer.astype(int)
    + has_carcinoma.astype(int)
    + has_neoplasm.astype(int)
)

# Articles containing ONLY one specific phrase
prostate_cancer_only = matches[
    has_cancer & (term_count == 1)
].copy()

prostate_carcinoma_only = matches[
    has_carcinoma & (term_count == 1)
].copy()

prostate_neoplasm_only = matches[
    has_neoplasm & (term_count == 1)
].copy()

# Articles containing two or all three phrases
multiple_terms = matches[
    term_count > 1
].copy()


# ==========================================
# EXPORT TO EXCEL
# ==========================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    # All qualifying articles
    matches.to_excel(
        writer,
        sheet_name="All Matches",
        index=False
    )

    # Contains ONLY "prostate cancer"
    prostate_cancer_only.to_excel(
        writer,
        sheet_name="Prostate Cancer",
        index=False
    )

    # Contains ONLY "prostate carcinoma"
    prostate_carcinoma_only.to_excel(
        writer,
        sheet_name="Prostate Carcinoma",
        index=False
    )

    # Contains ONLY "prostate neoplasm"
    prostate_neoplasm_only.to_excel(
        writer,
        sheet_name="Prostate Neoplasm",
        index=False
    )

    # Contains more than one of the requested phrases
    multiple_terms.to_excel(
        writer,
        sheet_name="Multiple Terms",
        index=False
    )

    # Articles with no abstract
    no_abstracts.to_excel(
        writer,
        sheet_name="No Abstract",
        index=False
    )


print("Saved to:")
print(OUTPUT_FILE)

print("\nBreakdown:")
print(f"All matching articles: {len(matches):,}")
print(f"Prostate cancer only: {len(prostate_cancer_only):,}")
print(f"Prostate carcinoma only: {len(prostate_carcinoma_only):,}")
print(f"Prostate neoplasm only: {len(prostate_neoplasm_only):,}")
print(f"Multiple terms: {len(multiple_terms):,}")
print(f"No abstract: {len(no_abstracts):,}")

# Sanity check:
exclusive_total = (
    len(prostate_cancer_only)
    + len(prostate_carcinoma_only)
    + len(prostate_neoplasm_only)
    + len(multiple_terms)
)

print(f"\nExclusive categories total: {exclusive_total:,}")
print(f"Matches total: {len(matches):,}")

assert exclusive_total == len(matches), \
    "Error: the mutually exclusive categories do not add up to All Matches."

Saved to:
/content/drive/MyDrive/AI Project/2023/2023 Prostate Abstract Matches.xlsx

Breakdown:
All matching articles: 5,588
Prostate cancer only: 5,528
Prostate carcinoma only: 31
Prostate neoplasm only: 1
Multiple terms: 28
No abstract: 809

Exclusive categories total: 5,588
Matches total: 5,588
